In [31]:
import tables
import numpy as np
from ctapipe_io_lst.constants import LST1_LOCATION as location
from datetime import datetime, timezone

import requests
from astropy.time import Time

a = tables.open_file("/data/cta/users-ifae/moralejo/CTA/summer_student_2026/datacheck/datacheck_dl1_LST-1.Run24704.h5")

In [32]:
# TEL RA AND DEC
tel_ra= np.degrees(a.root.dl1datacheck.cosmics.col('tel_ra')[subrun])
tel_dec = np.degrees(a.root.dl1datacheck.cosmics.col('tel_dec')[subrun])

# DRAGON TIMES (UTC for 50 events through the run, given in Unix time)
tmean_unix = np.mean(a.root.dl1datacheck.cosmics.col('dragon_time')[subrun])

# Conversion to datetime UTC
tmean_utc = datetime.fromtimestamp(tmean_unix, tz=timezone.utc)

# Conversion to Julian Date
obstime = Time(tmean_utc)

# Ask SatChecker API
url = "https://satchecker.astro.noirlab.edu/api/v1/fov"
params = {
    "latitude": location.lat.deg,
    "longitude": location.lon.deg,
    "elevation": location.height.to_value('m'),
    "julian_date": obstime.jd,
    "ra": tel_ra,
    "dec": tel_dec,
    "radius": 1.9  # LST1 Field of View radius in degrees
}

response = requests.get(url, params=params)

if response.status_code == 200:
    satellites = response.json().get("satellites", [])
    
    if not satellites:
        print("No satellites found in this subrun.")
    else:
        print(f"Satellites detected in FOV ({len(satellites)}):")
        for sat in satellites:
            print(f"- NORAD ID: {sat.get('norad_id')} | Name: {sat.get('name')}")
else:
    print(f"Error querying database: HTTP {response.status_code}")
    print("Details:", response.text)

ConnectionError: HTTPSConnectionPool(host='satchecker.astro.noirlab.edu', port=443): Max retries exceeded with url: /api/v1/fov?latitude=28.761526109999995&longitude=-17.891497010000023&elevation=2199.884999999286&julian_date=2461214.683213801&ra=18833.274645861085&dec=730.6986382404046&radius=1.9 (Caused by NameResolutionError("HTTPSConnection(host='satchecker.astro.noirlab.edu', port=443): Failed to resolve 'satchecker.astro.noirlab.edu' ([Errno -2] Name or service not known)"))

In [ ]:
# dragon time: UTC for 50 events through the run podemos calcular el tiempo medio de un subrun
# lo da en unix time , podemos convertirlo a tiempo normal con datetime y poner que lo convierta a  UTC timezome